In [23]:
import pyvista as pv
import numpy as np
import meshio
from pathlib import Path


def read_result(filename, timestep=None):
    """
    Read time-dependent result XDMF with PyVista.
    """

    filename = str(filename)
    reader = pv.get_reader(filename)

    if hasattr(reader, "time_values"):
        time_values = reader.time_values

        if len(time_values) > 0:
            if timestep is None:
                timestep = -1

            if timestep < 0:
                timestep = len(time_values) + timestep

            if timestep < 0 or timestep >= len(time_values):
                raise ValueError(
                    f"Invalid timestep {timestep}. "
                    f"Available steps: 0 to {len(time_values) - 1}"
                )

            reader.set_active_time_point(timestep)

            print(
                f"Reading timestep {timestep}: "
                f"time = {time_values[timestep]}"
            )

    mesh = reader.read()

    if isinstance(mesh, pv.MultiBlock):
        blocks = []

        for block in mesh:
            if block is not None and block.n_cells > 0:
                blocks.append(block)

        if not blocks:
            raise RuntimeError(
                f"No valid mesh blocks found in {filename}"
            )

        if len(blocks) == 1:
            mesh = blocks[0]
        else:
            mesh = pv.MultiBlock(blocks).combine()

    return mesh


def read_outline(filename):
    """
    Read static outline XDMF using meshio.

    This avoids the PyVista XDMF reader issue for XDMF files
    that contain no time information.
    """

    mesh = meshio.read(filename)

    points = np.asarray(mesh.points)

    line_cells = None

    for cell_block in mesh.cells:
        if cell_block.type == "line":
            line_cells = np.asarray(cell_block.data)
            break

    if line_cells is None:
        raise RuntimeError(
            "No line elements found in outline XDMF."
        )

    # PyVista line connectivity format:
    #
    # [2, p0, p1,
    #  2, p0, p1,
    #  ...]
    #
    lines = np.hstack(
        [
            np.full(
                (line_cells.shape[0], 1),
                2,
                dtype=np.int64,
            ),
            line_cells.astype(np.int64),
        ]
    ).ravel()

    outline = pv.PolyData(
        points,
        lines=lines,
    )

    return outline


def get_scalar_range(mesh, scalar_name):
    if scalar_name in mesh.point_data:
        values = mesh.point_data[scalar_name]

    elif scalar_name in mesh.cell_data:
        values = mesh.cell_data[scalar_name]

    else:
        available = (
            list(mesh.point_data.keys())
            + list(mesh.cell_data.keys())
        )

        raise KeyError(
            f"Scalar '{scalar_name}' not found.\n"
            f"Available arrays: {available}"
        )

    return np.nanmin(values), np.nanmax(values)


def visualize_damage(
    result_file,
    outline_file,
    timestep=-1,
    damage_name="damage",
    clip_value=0.0,
    invert=True,
    screenshot="damage_view.png",
    window_size=(1600, 1000),
    cmap="viridis",
    full_domain_opacity=0.3,
    outline_width=2.0,
    zoom=1.0,
):
    # -----------------------------------------------------------------
    # Read result
    # -----------------------------------------------------------------

    result = read_result(
        result_file,
        timestep=timestep,
    )

    # -----------------------------------------------------------------
    # Read separate outline
    # -----------------------------------------------------------------

    outline = read_outline(
        outline_file,
    )

    # -----------------------------------------------------------------
    # Damage range
    # -----------------------------------------------------------------

    damage_min, damage_max = get_scalar_range(
        result,
        damage_name,
    )

    print()
    print("Damage field")
    print("-------------------------------")
    print(f"Name       : {damage_name}")
    print(f"Minimum    : {damage_min}")
    print(f"Maximum    : {damage_max}")
    print(f"Clip value : {clip_value}")
    print(f"Invert     : {invert}")
    print("-------------------------------")
    print()

    # -----------------------------------------------------------------
    # Scalar clip
    # -----------------------------------------------------------------

    damage_clip = result.clip_scalar(
        scalars=damage_name,
        value=clip_value,
        invert=invert,
    )

    print(
        f"Full mesh:   "
        f"{result.n_points:,} points, "
        f"{result.n_cells:,} cells"
    )

    print(
        f"Damage clip: "
        f"{damage_clip.n_points:,} points, "
        f"{damage_clip.n_cells:,} cells"
    )

    print(
        f"Outline:     "
        f"{outline.n_points:,} points, "
        f"{outline.n_lines:,} lines"
    )

    # -----------------------------------------------------------------
    # Plotter
    # -----------------------------------------------------------------

    plotter = pv.Plotter(
        off_screen=True,
        window_size=window_size,
    )

    plotter.set_background("white")

    # -----------------------------------------------------------------
    # Full domain
    # -----------------------------------------------------------------

    plotter.add_mesh(
        result,
        # color="white",
        scalars=damage_name,
        cmap=cmap,
        opacity=full_domain_opacity,
        show_scalar_bar=False,
        show_edges=False,
        lighting=True,
    )

    # -----------------------------------------------------------------
    # Damage clip
    # -----------------------------------------------------------------

    plotter.add_mesh(
        damage_clip,
        scalars=damage_name,
        cmap=cmap,
        clim=(damage_min, damage_max),
        show_scalar_bar=False,
        lighting=True,
    )

    # -----------------------------------------------------------------
    # Outline
    # -----------------------------------------------------------------

    plotter.add_mesh(
        outline,
        color="black",
        line_width=outline_width,
        lighting=False,
    )

    # -----------------------------------------------------------------
    # Camera
    # -----------------------------------------------------------------

    bounds = result.bounds

    xmin, xmax = bounds[0], bounds[1]
    ymin, ymax = bounds[2], bounds[3]
    zmin, zmax = bounds[4], bounds[5]

    center = np.array(
        [
            0.5 * (xmin + xmax),
            0.5 * (ymin + ymax),
            0.5 * (zmin + zmax),
        ]
    )

    dx = xmax - xmin
    dy = ymax - ymin
    dz = zmax - zmin

    diagonal = np.sqrt(
        dx**2
        + dy**2
        + dz**2
    )

    direction = np.array(
        [
            1.0,
            -1.0,
            1.0,
        ]
    )

    direction /= np.linalg.norm(
        direction
    )

    camera_distance = 2.0 * diagonal

    camera_position = (
        center
        + camera_distance * direction
    )

    plotter.camera_position = [
        camera_position,
        center,
        (0.0, 0.0, 1.0),
    ]

    plotter.camera.up = (
        0.0,
        0.0,
        1.0,
    )

    plotter.reset_camera()
    # Rotate view +90 degrees about Z
    plotter.camera.Azimuth(-90)
    if zoom != 1.0:
        plotter.camera.zoom(zoom)

    # -----------------------------------------------------------------
    # Screenshot
    # -----------------------------------------------------------------

    screenshot = Path(screenshot)

    screenshot.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plotter.screenshot(
        str(screenshot)
    )

    plotter.close()

    print()
    print(
        f"Screenshot written to: "
        f"{screenshot}"
    )


# =====================================================================
# USER SETTINGS
# =====================================================================

result_file = "06/output.xdmf"

outline_file = "../../../data/mesh/02/Lx500_1C_AD_outline.xdmf"

# -1 = last step
#  0 = first step
#  1 = second step
step = -1

damage_variable = "damage"

damage_clip_value = 0.99


# =====================================================================
# RUN
# =====================================================================

visualize_damage(
    result_file=result_file,
    outline_file=outline_file,
    timestep=step,
    damage_name=damage_variable,
    clip_value=damage_clip_value,
    invert=False,
    screenshot=f"06/damage_step_{step}.png",
    window_size=(1600, 1000),
    cmap="jet",
    full_domain_opacity=0.05,
    outline_width=2.0,
    zoom=1.0,
)

Reading timestep 27: time = 27.0

Damage field
-------------------------------
Name       : damage
Minimum    : -1.4484152555846454e-17
Maximum    : 1.0
Clip value : 0.99
Invert     : False
-------------------------------

Full mesh:   170,605 points, 943,190 cells
Damage clip: 91,631 points, 271,547 cells
Outline:     911 points, 919 lines

Screenshot written to: 06/damage_step_-1.png
